# TP 2 — Apprentissage supervise 2A (correction R)

**Objectif du sujet.** Etudier un **unique** jeu de donnees (vins) et y appliquer tous les modeles de
**discrimination** vus en cours : **Naive Bayes**, **analyse discriminante lineaire/quadratique (LDA/QDA)** et
**k plus proches voisins (k-NN)**. On construit un jeu d'apprentissage et un jeu de test, on regle les
hyperparametres, puis on **compare** les modeles sur le test.

> Correction en **R** (`MASS`, `e1071`, `class`/`caret`, `pROC`). Le jeu « wine quality (red) » est charge
> depuis le depot **UCI**. On transforme la note en **classe binaire** : $Y = 1$ (**bon vin**) si `quality` $\ge 7$ (note de **7 ou 8**),
> sinon $Y = 0$ (**mauvais vin**, note $\le 6$).


## Brief de demarrage — Les outils de ce TP

> **A lire avant de commencer.** Ce TP compare quatre classifieurs. Voici, en quelques lignes, **comment
> fonctionne chacun** et **ce qu'il faut retenir** avant de coder. Le point commun de tous ces modeles : ils
> cherchent a approcher $\eta(x) = \mathbb{P}(Y = 1 \mid X = x)$ puis decident via la **regle de Bayes** (seuil
> $1/2$). Une illustration graphique de chaque frontiere est donnee dans les **annexes** en fin de notebook.

**1. Naive Bayes gaussien — un modele *generatif*.**
On modelise chaque classe separement. Par la regle de Bayes,
$\mathbb{P}(Y=c\mid x) \propto \mathbb{P}(Y=c)\prod_j \mathbb{P}(x_j \mid Y=c)$. L'hypothese **« naive »** est que
les variables sont **independantes conditionnellement a la classe**, et chaque $x_j\mid Y=c$ suit une gaussienne
$\mathcal{N}(\mu_{cj}, \sigma_{cj}^2)$. Concretement : on ajuste une cloche par variable et par classe, puis on
multiplie. Tres **rapide** et **robuste**, mais l'hypothese d'independance est souvent fausse.

**2. Analyse discriminante LDA / QDA — modeles *gaussiens multivaries*.**
On suppose $X \mid Y=c \sim \mathcal{N}(\mu_c, \Sigma_c)$ (une « ellipse » de densite par classe).
- **LDA** : on impose la **meme** matrice de covariance a toutes les classes ($\Sigma_c = \Sigma$). La frontiere
  de decision est alors une **droite** (un hyperplan).
- **QDA** : chaque classe a **sa propre** covariance. La frontiere devient une **courbe** (quadratique), donc
  plus flexible — au prix de beaucoup plus de parametres a estimer.

**3. k plus proches voisins (k-NN) — modele *non parametrique*.**
Aucune hypothese de forme : pour classer un point, on regarde ses **$k$ voisins** les plus proches et on vote.
$k$ petit ⇒ frontiere tres decoupee (variance forte) ; $k$ grand ⇒ frontiere lisse (biais fort). Le k-NN
raisonne par **distances**, il faut donc **standardiser** les variables au prealable.

| Modele | Hypothese cle | Frontiere | Hyperparametre | Standardisation |
|---|---|---|---|---|
| Naive Bayes | variables independantes / classe | quadratique (diagonale) | lissage | inutile |
| LDA | gaussiennes, **covariances egales** | **lineaire** | `shrinkage` | inutile |
| QDA | gaussiennes, covariances libres | **quadratique** | regularisation | inutile |
| k-NN | aucune (local) | tres flexible | $k$ | **indispensable** |

> **Methode du TP.** (1) EDA pour comprendre les donnees ; (2) decoupage **stratifie** apprentissage/test ;
> (3) reglage des hyperparametres par **validation croisee** (on optimise l'**AUC**) ; (4) comparaison finale
> sur le test via **accuracy** et **AUC** (courbes ROC).

In [1]:
source(if (file.exists("plot_theme.R")) "plot_theme.R" else "../plot_theme.R")  # theme + charger_packages()
charger_packages(c(
                   "MASS",    # lda, qda
                   "e1071",
                   "klaR",
                   "class",
                   "caret",
                   "pROC",
                   "plotly"
))

SEED <- 0
set.seed(SEED)   # graine unique posee une fois : tout le TP est reproductible

## Partie 1 — Chargement et analyse exploratoire

### Rappel de cours — Pourquoi une analyse exploratoire (EDA) ?

Avant tout modele, on **regarde les donnees** pour :
- reperer les **echelles** et **distributions** des variables (utile pour choisir les pretraitements) ;
- detecter un eventuel **desequilibre des classes** (impacte l'echantillonnage et les metriques) ;
- identifier les variables **discriminantes** (correlees a la cible).

### Question 1 — Chargement et resume du tableau


In [2]:

url <- "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
wine <- read.csv(url, sep = ";")

quality <- wine$quality                       # note de 3 a 8
X <- wine[, setdiff(names(wine), "quality")]  # 11 variables explicatives
# Cible binaire : bon vin (note de 7 ou 8) vs mauvais vin (note <= 6)
y <- factor(ifelse(quality >= 7, "bon", "mauvais"), levels = c("mauvais", "bon"))

cat("Dimensions des predicteurs :", dim(X), "\n\n")
print(round(t(sapply(X, summary)), 2))


Dimensions des predicteurs : 1599 11 



                     Min. 1st Qu. Median  Mean 3rd Qu.   Max.
fixed.acidity        4.60    7.10   7.90  8.32    9.20  15.90
volatile.acidity     0.12    0.39   0.52  0.53    0.64   1.58
citric.acid          0.00    0.09   0.26  0.27    0.42   1.00
residual.sugar       0.90    1.90   2.20  2.54    2.60  15.50
chlorides            0.01    0.07   0.08  0.09    0.09   0.61
free.sulfur.dioxide  1.00    7.00  14.00 15.87   21.00  72.00
total.sulfur.dioxide 6.00   22.00  38.00 46.47   62.00 289.00
density              0.99    1.00   1.00  1.00    1.00   1.00
pH                   2.74    3.21   3.31  3.31    3.40   4.01
sulphates            0.33    0.55   0.62  0.66    0.73   2.00
alcohol              8.40    9.50  10.20 10.42   11.10  14.90


In [3]:

cat("Repartition de la note 'quality' :\n"); print(table(quality))
cat("\nRepartition de la classe binaire :\n"); print(table(y))
cat("\nProportion de bons vins :", round(mean(y == "bon"), 3), "\n")


Repartition de la note 'quality' :


quality
  3   4   5   6   7   8 
 10  53 681 638 199  18 



Repartition de la classe binaire :


y
mauvais     bon 
   1382     217 



Proportion de bons vins : 0.136 


In [4]:

qc <- as.data.frame(table(quality)); names(qc) <- c("quality", "Effectif")
plot_ly(qc, x = ~quality, y = ~Effectif, type = "bar",
        text = ~Effectif, textposition = "outside",
        marker = list(color = pick_color(1))) %>%
  layout(title = "Distribution de la note quality",
         xaxis = list(title = "quality"), yaxis = list(title = "Effectif")) %>%
  apply_dark_theme()


HTML widgets cannot be represented in plain text (need html)

In [5]:
# Distribution de la cible binarisee : mauvais vin (note <= 6) / bon vin (note 7-8)
bc <- as.data.frame(table(y)); names(bc) <- c("classe", "Effectif")
plot_ly(bc, x = ~classe, y = ~Effectif, type = "bar", text = ~Effectif, textposition = "outside",
        marker = list(color = c(pick_class_color("mauvais"), pick_class_color("bon")))) %>%
  layout(title = "Distribution de la cible binarisee : mauvais vin (<= 6) / bon vin (>= 7)",
         xaxis = list(title = "Classe"), yaxis = list(title = "Effectif")) %>%
  apply_dark_theme()

HTML widgets cannot be represented in plain text (need html)

In [6]:

vars <- names(X)
n_cols <- 3; n_rows <- ceiling(length(vars) / n_cols)
plots <- lapply(seq_along(vars), function(i) {
  v <- vars[i]
  plot_ly(x = X[[v]], type = "histogram", nbinsx = 25,
          marker = list(color = pick_color(i - 1)), name = v) %>%
    layout(xaxis = list(title = v), yaxis = list(title = "Effectif"))
})
subplot(plots, nrows = n_rows, titleX = TRUE, margin = 0.05) %>%
  layout(title = "Distribution de chaque variable explicative", showlegend = FALSE) %>%
  apply_dark_theme(height = 250 * n_rows)


HTML widgets cannot be represented in plain text (need html)

**Ce que l'on remarque (reponse Q1) :**
- Les variables sont sur des **echelles tres differentes** (ex. `total.sulfur.dioxide` jusqu'a ~289,
  `density` ~ 0.99, `chlorides` ~ 0.08). ⇒ une **standardisation** sera necessaire pour le k-NN.
- Plusieurs variables sont **asymetriques** / presentent des **valeurs extremes**.
- Avec le seuil « bon vin = note $\ge 7$ », les classes sont **desequilibrees** : seuls **~14 %** des vins sont
  « bons ». ⇒ decoupage **stratifie** indispensable, et comparaison des modeles fondee sur l'**AUC**.
- Aucune valeur manquante.

### Question 2 — Exploration visuelle : quelles variables influencent la qualite ?

> **La cible est binarisee des le depart** (`y` = bon/mauvais). On cherche donc la correlation entre une variable
> **continue** (ex. `alcohol`) et une variable **binaire** (`y`). Le coefficient adapte n'est pas la correlation
> de Pearson usuelle entre deux variables continues, mais la **correlation point-bisereale** : c'est le **cas
> particulier de Pearson** obtenu lorsqu'une des deux variables est binaire (codee 0/1). Elle mesure a quel point
> la variable continue **separe** les deux classes.

On regarde donc la **correlation point-bisereale** de chaque variable avec la classe, puis la distribution des
variables les plus correlees selon la classe (**boxplots**).


In [7]:
# On compare une variable CONTINUE a la cible BINAIRE (0/1) : le coefficient adapte est
# la correlation point-bisereale (cas particulier de Pearson quand une variable est binaire).
yb <- as.numeric(y == "bon")
corr <- sapply(X, function(col) cor(col, yb))
corr <- sort(corr)
cdf <- data.frame(variable = factor(names(corr), levels = names(corr)),
                  correlation = as.numeric(corr))
cdf$signe <- ifelse(cdf$correlation >= 0, "positive", "negative")

plot_ly(cdf, x = ~correlation, y = ~variable, type = "bar", orientation = "h",
        color = ~signe, colors = c(negative = pick_color(1), positive = pick_color(0))) %>%
  layout(title = "Correlation point-bisereale de chaque variable avec la classe (bon/mauvais)",
         xaxis = list(title = "Correlation point-bisereale"), yaxis = list(title = "")) %>%
  apply_dark_theme()


HTML widgets cannot be represented in plain text (need html)

In [8]:

top_vars <- c("alcohol", "volatile.acidity", "sulphates", "citric.acid")
mk_box <- function(v) {
  df <- data.frame(classe = y, valeur = X[[v]])
  plot_ly(df, x = ~classe, y = ~valeur, type = "box", color = ~classe,
          colors = c(mauvais = pick_class_color("mauvais"), bon = pick_class_color("bon"))) %>%
    layout(xaxis = list(title = ""), yaxis = list(title = v))
}
subplot(lapply(top_vars, mk_box), nrows = 1, titleX = TRUE, margin = 0.05) %>%
  layout(title = "Variables les plus correlees selon la classe", showlegend = FALSE) %>%
  apply_dark_theme()


HTML widgets cannot be represented in plain text (need html)

**Ce que l'on remarque (reponse Q2) :** les variables qui semblent **le plus influencer** la qualite sont :
- **`alcohol`** : forte correlation **positive** (les bons vins sont plus alcoolises) — la plus discriminante ;
- **`volatile.acidity`** : correlation **negative** (trop d'acidite volatile = mauvais vin) ;
- **`sulphates`** et **`citric.acid`** : correlation positive plus moderee.


## Partie 2 — Construction des echantillons et entrainement des modeles

### Rappel de cours — Les modeles de discrimination

Tous estiment (explicitement ou non) $\eta(x) = \mathbb{P}(Y = 1 \mid X = x)$, puis appliquent la regle de
Bayes (seuil $1/2$).

- **Naive Bayes gaussien** : modele **generatif**,
  $\mathbb{P}(Y=c\mid x) \propto \mathbb{P}(Y=c)\prod_j \mathbb{P}(x_j \mid Y=c)$, avec l'hypothese **naive**
  d'**independance** des variables conditionnellement a la classe. Rapide, robuste, hypothese forte.
- **Analyse discriminante (LDA / QDA)** : $X \mid Y=c \sim \mathcal{N}(\mu_c, \Sigma_c)$.
  - **LDA** : covariances **egales** ⇒ frontiere **lineaire**.
  - **QDA** : covariances **differentes** ⇒ frontiere **quadratique** (plus flexible).
- **k-NN** : **non parametrique**, vote des $k$ voisins. Hyperparametre **$k$** ; sensible a l'echelle ⇒ **standardiser**.

> **LDA/QDA/Naive Bayes** sont **invariants** a une mise a l'echelle par variable ; le **k-NN**, lui, l'exige.

### Question 1 — Decoupage apprentissage / test

Classes non parfaitement equilibrees ⇒ decoupage **stratifie** (conserve la proportion de bons/mauvais vins).


In [9]:
idx <- createDataPartition(y, p = 0.75, list = FALSE)   # decoupage stratifie
Xtr <- X[idx, ];  Xte <- X[-idx, ]
ytr <- y[idx];    yte <- y[-idx]
cat("Apprentissage :", nrow(Xtr), "| Test :", nrow(Xte), "\n")
cat("Proportion de bons vins  -> train :", round(mean(ytr == "bon"), 3),
    "| test :", round(mean(yte == "bon"), 3), "\n")

Apprentissage : 1200 | Test : 399 


Proportion de bons vins  -> train : 0.136 | test : 0.135 


### Question 2 — Entrainement et reglage des hyperparametres

On regle les hyperparametres par **validation croisee** (5 plis) en optimisant l'**AUC** (voir **Annexe H** pour
le choix AUC vs accuracy). **Ne pas oublier d'hyperparametres !** Le Naive Bayes, en particulier, en possede
plusieurs sous `caret` (methode `nb`) :

- **Naive Bayes** : `usekernel` (densite par **noyau** au lieu d'une gaussienne), `adjust` (largeur de la
  fenetre du noyau) et `fL` (correction de Laplace). Le principe du noyau est detaille dans l'**Annexe F**.
- **LDA / QDA** : pas d'hyperparametre a proprement parler (eventuellement une regularisation de la covariance) ;
- **k-NN** : le nombre de voisins `k`, precede d'une **standardisation**.

> Le **role precis** de chaque hyperparametre — et les **autres reglages disponibles** pour LDA, QDA et Naive
> Bayes — sont detailles dans l'**Annexe G**.


In [10]:
# Naive Bayes : on regle plusieurs hyperparametres (usekernel / fL / adjust) par validation croisee
ctrl <- trainControl(method = "cv", number = 5,
                     classProbs = TRUE, summaryFunction = twoClassSummary)
grille_nb <- expand.grid(usekernel = c(TRUE, FALSE),
                         fL = c(0, 0.5, 1),
                         adjust = c(0.5, 1, 1.5))
nb_fit <- suppressWarnings(train(Xtr, ytr, method = "nb", metric = "ROC",
                                 trControl = ctrl, tuneGrid = grille_nb))

# LDA / QDA : pas (ou quasi pas) d'hyperparametre a regler
lda_m <- lda(Xtr, ytr)
qda_m <- qda(Xtr, ytr)

# k-NN : on regle k par validation croisee (standardisation prealable)
knn_fit <- train(Xtr, ytr, method = "knn", metric = "ROC",
                 preProcess = c("center", "scale"),
                 tuneGrid = data.frame(k = seq(1, 81, 2)), trControl = ctrl)

cat("Naive Bayes : meilleurs params ->",
    paste(names(nb_fit$bestTune), unlist(nb_fit$bestTune), collapse = ", "),
    "| AUC CV =", round(max(nb_fit$results$ROC, na.rm = TRUE), 3), "\n")
cat("k-NN       : meilleur k =", knn_fit$bestTune$k,
    "| AUC CV =", round(max(knn_fit$results$ROC), 3), "\n")
cat("Note : LDA/QDA/Naive Bayes sont invariants a une mise a l'echelle par variable ;\n")
cat("       le k-NN, lui, exige une standardisation (preProcess center/scale).\n")

Naive Bayes : meilleurs params -> fL 0, usekernel 1, adjust 1 | AUC CV = 0.875 


k-NN       : meilleur k = 81 | AUC CV = 0.843 


Note : LDA/QDA/Naive Bayes sont invariants a une mise a l'echelle par variable ;


       le k-NN, lui, exige une standardisation (preProcess center/scale).


**Visualiser l'erreur selon les hyperparametres.** Pour le Naive Bayes, on trace l'**AUC de validation croisee**
selon `adjust` (largeur du noyau) et `usekernel` ; pour le k-NN, selon `k`. On y lit le compromis biais-variance :
un `k` trop petit sur-apprend, un `k` trop grand sous-apprend ; le pic de chaque courbe donne la valeur retenue.


In [11]:
# Naive Bayes : AUC (CV) selon 'adjust', une courbe par 'usekernel'
nb_res <- nb_fit$results
p_nb <- plot_ly()
for (uk in unique(nb_res$usekernel)) {
  d <- aggregate(ROC ~ adjust, data = nb_res[nb_res$usekernel == uk, ], FUN = max)
  p_nb <- add_trace(p_nb, data = d, x = ~adjust, y = ~ROC, type = "scatter",
                    mode = "lines+markers", name = paste("usekernel =", uk))
}
p_nb <- layout(p_nb, xaxis = list(title = "adjust (largeur du noyau)"),
               yaxis = list(title = "AUC (CV)"))

# k-NN : AUC (CV) selon k
p_knn <- plot_ly(knn_fit$results, x = ~k, y = ~ROC, type = "scatter",
                 mode = "lines+markers", name = "k-NN",
                 line = list(color = pick_color(3))) %>%
  layout(xaxis = list(title = "k (n_neighbors)"), yaxis = list(title = "AUC (CV)"))

subplot(p_nb, p_knn, nrows = 1, titleX = TRUE, margin = 0.07) %>%
  layout(title = "Erreur de validation croisee (AUC) selon les hyperparametres : Naive Bayes (gauche), k-NN (droite)") %>%
  apply_dark_theme(height = 420)


HTML widgets cannot be represented in plain text (need html)

### Question 3 — Comparaison sur le jeu de test

On evalue chaque modele **une seule fois** sur le test, avec trois metriques :
- l'**accuracy** (taux de bonnes predictions au seuil $1/2$) ;
- le **recall** de la classe « bon », c.-a-d. la part des **bons vins reellement detectes** ;
- l'**AUC** (aire sous la courbe ROC), qui mesure la capacite a **ordonner** les vins par probabilite d'etre
  bons, independamment du seuil (voir **Annexe H** pour le detail du choix AUC vs accuracy).

In [12]:
# Probabilites predites de la classe "bon" et predictions au seuil 1/2
proba <- suppressWarnings(list(
  "Naive Bayes" = predict(nb_fit, Xte, type = "prob")[, "bon"],
  "LDA"         = predict(lda_m, Xte)$posterior[, "bon"],
  "QDA"         = predict(qda_m, Xte)$posterior[, "bon"],
  "k-NN"        = predict(knn_fit, Xte, type = "prob")[, "bon"]
))
pred <- suppressWarnings(list(
  "Naive Bayes" = predict(nb_fit, Xte),
  "LDA"         = predict(lda_m, Xte)$class,
  "QDA"         = predict(qda_m, Xte)$class,
  "k-NN"        = predict(knn_fit, Xte)
))

roc_df <- function(p, name) {
  r <- roc(yte, p, levels = c("mauvais", "bon"), direction = "<", quiet = TRUE)
  d <- data.frame(fpr = 1 - r$specificities, tpr = r$sensitivities)
  d <- d[order(d$fpr, d$tpr), ]
  d$modele <- name
  d
}

fig <- plot_ly()
resultats <- data.frame()
noms <- names(proba)
for (i in seq_along(noms)) {
  nm <- noms[i]
  a  <- as.numeric(auc(roc(yte, proba[[nm]], levels = c("mauvais","bon"), direction = "<", quiet = TRUE)))
  ac <- mean(pred[[nm]] == yte)
  rc <- mean(pred[[nm]][yte == "bon"] == "bon")   # recall de la classe "bon"
  resultats <- rbind(resultats, data.frame(Modele = nm, Accuracy = round(ac, 3),
                                           Recall = round(rc, 3), AUC = round(a, 3)))
  d <- roc_df(proba[[nm]], nm)
  fig <- add_trace(fig, data = d, x = ~fpr, y = ~tpr, type = "scatter", mode = "lines",
                   name = sprintf("%s (AUC=%.3f)", nm, a),
                   line = list(width = 3, color = pick_color(i - 1)))
}
fig <- add_trace(fig, x = c(0, 1), y = c(0, 1), type = "scatter", mode = "lines",
                 name = "Aleatoire", line = list(color = MUTED, dash = "dash"))
fig <- layout(fig, title = "Courbes ROC sur le jeu de test - comparaison des modeles",
              xaxis = list(title = "Taux de faux positifs"),
              yaxis = list(title = "Taux de vrais positifs"))
fig <- apply_dark_theme(fig)

resultats <- resultats[order(-resultats$AUC), ]
print(resultats, row.names = FALSE)
fig

      Modele Accuracy Recall   AUC
 Naive Bayes    0.865  0.648 0.889
         LDA    0.872  0.333 0.877
        k-NN    0.870  0.074 0.862
         QDA    0.842  0.630 0.835


HTML widgets cannot be represented in plain text (need html)

### Conclusion — Quel est le modele le plus predictif ?

On classe les modeles par **AUC** sur le test (tableau ci-dessus). Les quatre modeles sont **proches** (AUC entre
~0.83 et ~0.89) :
- le **Naive Bayes a noyau** (`usekernel = TRUE`) arrive en tete : l'estimation par noyau capte des densites
  asymetriques que la gaussienne represente mal (voir **Annexe F**) ;
- la **LDA** suit de pres : une frontiere lineaire suffit deja a bien ordonner les vins ;
- le **k-NN** choisit un $k$ **tres grand** (en bord de grille) : avec l'AUC comme critere, un grand $k$ donne un
  score (proportion de voisins « bons ») plus fin et plus lisse, donc un meilleur ordonnancement... mais au seuil
  $1/2$ il ne predit presque jamais « bon » (recall tres faible) ;
- la **QDA** est en retrait : elle estime une covariance par classe (beaucoup de parametres) sur une classe
  minoritaire de ~160 vins, d'ou une variance d'estimation plus forte.

Le **recall** montre un autre visage : au seuil $1/2$, Naive Bayes et QDA detectent environ **deux tiers** des
bons vins, LDA un tiers, le k-NN presque aucun. Avec des classes desequilibrees, **une seule metrique ne suffit
pas** : l'AUC juge l'ordonnancement, le recall la detection de la classe rare, et l'accuracy est trompeuse (un
classifieur predisant « tout mauvais » atteint deja ~86 %).

**Le modele retenu** est celui qui maximise l'AUC (premiere ligne du tableau) ; le seuil de decision pourra
ensuite etre abaisse si l'on veut detecter davantage de bons vins (voir TP4).

## Annexe A — Pourquoi la règle de Bayes est le point de comparaison central

Le TP2 compare plusieurs classifieurs : Naive Bayes, LDA/QDA et k-NN. Ils ont tous en commun le fait qu’ils cherchent à approcher la fonction
$$\eta(x) = \mathbb{P}(Y=1 \mid X=x).$$

La règle de Bayes
$$g^\star(x)=\mathbb{1}\{\eta(x) \ge 1/2\}$$
est la meilleure décision possible pour le coût 0–1. C’est donc la référence absolue utilisée pour juger la qualité d’un modèle.

### Pourquoi cela aide à interpréter les résultats ?
- Si un modèle a une erreur proche du risque de Bayes, alors il est presque optimal.
- Si son erreur reste nettement plus élevée, cela signifie qu’il reste du bruit, une mauvaise spécification ou un manque d’information.
- Le but n’est pas de « battre » Bayes : c’est de s’en rapprocher autant que possible à partir d’un échantillon fini.

---

## Annexe B — Naive Bayes, LDA/QDA et les frontières de décision

### Naive Bayes : une approximation très utile
Le modèle suppose que les variables sont indépendantes conditionnellement à la classe :
$$\mathbb{P}(X_1,\dots,X_p \mid Y=c) = \prod_j \mathbb{P}(X_j \mid Y=c).$$
Cette hypothèse est souvent fausse, mais elle rend le calcul très simple et souvent robuste.

En pratique, cela revient à dire : pour chaque classe, on estime la densité conditionnelle de chaque variable puis on multiplie les contributions. La règle de décision est alors obtenue en comparant les scores de vraisemblance pour chaque classe.

### LDA / QDA : frontiers linéaires ou quadratiques
Lorsque l’on suppose
$$X \mid Y=c \sim \mathcal{N}(\mu_c, \Sigma_c),$$
les densités conditionnelles sont gaussiennes. On obtient alors une règle de décision qui compare des scores de vraisemblance.

- **LDA** : covariances égales ⇒ frontière de décision **linéaire**.
- **QDA** : covariances différentes ⇒ frontière **quadratique**.

Ainsi, le choix de LDA ou QDA dépend du degré de flexibilité qu’on laisse à la séparation entre classes.

---

## Annexe C — Pourquoi le k-NN exige une standardisation

Le k-NN mesure des distances dans l’espace des variables. Si une variable a une échelle beaucoup plus grande qu’une autre, elle domine la distance et masque les autres informations.

Exemple : si une variable est mesurée en milliers et une autre en unités, la première « écrase » la seconde dans la distance euclidienne. C’est pourquoi on standardise :
$$x_j^{(std)} = \frac{x_j - \mu_j}{\sigma_j}.$$

Cette transformation remet toutes les variables à la même échelle. Elle est cruciale pour les méthodes basées sur les distances, comme le k-NN, et moins nécessaire pour des méthodes comme Naive Bayes ou LDA/QDA.

**Interprétation.** Le k-NN n’est pas seulement un algorithme de vote : il se comporte comme un estimateur local de $\eta(x)$. Le choix de $k$ et l’échelle des variables influencent directement la qualité de cette estimation locale.

---

## Annexe D — Illustration graphique : les frontieres de decision

Pour bien **visualiser** ce que font LDA, QDA et Naive Bayes, on les applique a un jeu **synthetique en 2D** :
deux classes gaussiennes ayant des **covariances differentes**. Chaque panneau montre la **probabilite predite**
$\hat{\mathbb{P}}(Y=1\mid x)$ (fond colore : rouge = classe 0, bleu = classe 1) ; la **frontiere de decision**
se situe la ou la couleur bascule (seuil $1/2$).

Ce que l'on doit observer :
- **LDA** : covariances supposees **egales** ⇒ frontiere **rigoureusement lineaire** (une droite) ;
- **QDA** : covariances **libres** par classe ⇒ frontiere **quadratique** (courbee), qui epouse mieux la forme
  des deux nuages ;
- **Naive Bayes** : gaussiennes a axes **alignes** (independance des variables) ⇒ frontiere quadratique mais
  contrainte, souvent proche de la QDA quand les variables sont peu correlees.

In [13]:
# Jeu synthetique 2D : deux classes gaussiennes de covariances differentes
n_pts <- 300
X0 <- MASS::mvrnorm(n_pts, mu = c(-1.5, -1.0), Sigma = matrix(c(1.3, 0.7, 0.7, 0.6), 2))
X1 <- MASS::mvrnorm(n_pts, mu = c( 1.6,  1.1), Sigma = matrix(c(0.7, -0.6, -0.6, 1.5), 2))
dat <- data.frame(x1 = c(X0[, 1], X1[, 1]), x2 = c(X0[, 2], X1[, 2]),
                  classe = factor(rep(c("classe 0", "classe 1"), each = n_pts)))

# Grille d'evaluation de la probabilite predite P(Y = classe 1 | x)
gx <- seq(min(dat$x1) - 1, max(dat$x1) + 1, length.out = 160)
gy <- seq(min(dat$x2) - 1, max(dat$x2) + 1, length.out = 160)
grille <- expand.grid(x1 = gx, x2 = gy)

lda_i <- lda(classe ~ x1 + x2, data = dat)
qda_i <- qda(classe ~ x1 + x2, data = dat)
nb_i  <- naiveBayes(classe ~ x1 + x2, data = dat)

proba <- list(
  "LDA - frontiere lineaire"    = predict(lda_i, grille)$posterior[, "classe 1"],
  "QDA - frontiere quadratique" = predict(qda_i, grille)$posterior[, "classe 1"],
  "Naive Bayes - axes alignes"  = predict(nb_i, grille, type = "raw")[, "classe 1"]
)

faire_panneau <- function(p, titre) {
  z <- matrix(p, nrow = length(gy), ncol = length(gx), byrow = TRUE)
  plot_ly() %>%
    add_trace(x = gx, y = gy, z = z, type = "heatmap", zmin = 0, zmax = 1,
              colorscale = DIVERGING_SCALE, opacity = 0.8,
              showscale = FALSE, hoverinfo = "skip") %>%
    add_trace(x = X0[, 1], y = X0[, 2], type = "scatter", mode = "markers",
              name = "classe 0", showlegend = FALSE,
              marker = list(color = pick_class_color(0), size = 4,
                            line = list(color = BACKGROUND, width = 0.4))) %>%
    add_trace(x = X1[, 1], y = X1[, 2], type = "scatter", mode = "markers",
              name = "classe 1", showlegend = FALSE,
              marker = list(color = pick_class_color(1), size = 4,
                            line = list(color = BACKGROUND, width = 0.4))) %>%
    layout(xaxis = list(title = titre), yaxis = list(title = "x2"))
}

panneaux <- Map(faire_panneau, proba, names(proba))
subplot(panneaux, nrows = 1, shareY = TRUE, titleX = TRUE, margin = 0.04) %>%
  layout(title = "Frontieres de decision : LDA (droite) vs QDA (courbe) vs Naive Bayes") %>%
  apply_dark_theme(height = 420)

HTML widgets cannot be represented in plain text (need html)

---

## Annexe E — Demonstration : pourquoi la frontiere est **lineaire** (LDA) ou **quadratique** (QDA)

> **Message central a retenir.** On ne suppose **aucune relation lineaire entre les variables explicatives**.
> L'hypothese porte uniquement sur la **loi de $X$ dans chaque classe** (gaussienne) et sur le fait que les
> covariances sont **egales** (LDA) ou **libres** (QDA). C'est **la frontiere de decision** (dans l'espace des
> $x$) qui herite de cette forme droite ou courbe — pas une pretendue relation entre les variables.

### Point de depart commun

En classification binaire $Y \in \{0,1\}$, on suppose par classe $X \mid Y=c \sim \mathcal{N}(\hat\mu_c, \hat\Sigma_c)$
et on note $\hat\pi_c$ la proportion de la classe $c$. Par la regle de Bayes :
$$\hat{\mathbb{P}}(Y=c \mid X=x) \;\propto\; \mathcal{N}_x(\hat\mu_c, \hat\Sigma_c)\,\hat\pi_c .$$
La **frontiere de decision** est l'ensemble des $x$ tels que $\hat{\mathbb{P}}(Y=1\mid x) = \hat{\mathbb{P}}(Y=0\mid x)$.

### Cas LDA : covariances egales $\hat\Sigma_0 = \hat\Sigma_1 = \hat\Sigma$

En prenant le logarithme de l'egalite $\mathcal{N}_x(\hat\mu_1,\hat\Sigma)\hat\pi_1 = \mathcal{N}_x(\hat\mu_0,\hat\Sigma)\hat\pi_0$ :
$$-\tfrac12 (x-\hat\mu_1)^\top \hat\Sigma^{-1}(x-\hat\mu_1) + \log\hat\pi_1
  = -\tfrac12 (x-\hat\mu_0)^\top \hat\Sigma^{-1}(x-\hat\mu_0) + \log\hat\pi_0 .$$
En developpant, le terme quadratique $x^\top \hat\Sigma^{-1} x$ est **identique des deux cotes** et **s'annule** :
$$x^\top \hat\Sigma^{-1}(\hat\mu_1-\hat\mu_0)
  \;-\; \tfrac12 \hat\mu_1^\top \hat\Sigma^{-1}\hat\mu_1
  \;+\; \tfrac12 \hat\mu_0^\top \hat\Sigma^{-1}\hat\mu_0
  \;+\; \log\frac{\hat\pi_1}{\hat\pi_0} \;=\; 0 .$$
C'est une equation de la forme $x^\top w + b = 0$ : **l'equation d'un hyperplan**. La frontiere est donc
**lineaire** — d'ou le nom « analyse discriminante *lineaire* ».

### Cas QDA : covariances libres $\hat\Sigma_0 \neq \hat\Sigma_1$

Le meme calcul, sans pouvoir simplifier les covariances, donne :
$$-\tfrac12\, x^\top (\hat\Sigma_1^{-1} - \hat\Sigma_0^{-1})\, x
  \;+\; x^\top (\hat\Sigma_1^{-1}\hat\mu_1 - \hat\Sigma_0^{-1}\hat\mu_0)
  \;-\; \tfrac12 \hat\mu_1^\top \hat\Sigma_1^{-1}\hat\mu_1
  \;+\; \tfrac12 \hat\mu_0^\top \hat\Sigma_0^{-1}\hat\mu_0
  \;+\; \log\frac{\hat\pi_1}{\hat\pi_0}
  \;-\; \tfrac12 \log\frac{\det\hat\Sigma_1}{\det\hat\Sigma_0} \;=\; 0 .$$
C'est cette fois de la forme $x^\top A x + x^\top w + b = 0$ : **l'equation d'une surface quadratique**. La
frontiere est donc **quadratique** (courbee) — d'ou « analyse discriminante *quadratique* ».

### La conclusion cle

Le **terme quadratique** $-\tfrac12\, x^\top (\hat\Sigma_1^{-1} - \hat\Sigma_0^{-1})\, x$ **disparait exactement
lorsque $\hat\Sigma_0 = \hat\Sigma_1$** : la QDA **redonne** alors la LDA. La forme de la frontiere (droite ou
courbe) est donc une **consequence mathematique** de l'hypothese faite sur les covariances, et non une hypothese
de linearite entre les variables.


---

## Annexe F — Le principe du **noyau** (kernel)

### A quoi ca sert ?

Le Naive Bayes gaussien suppose que chaque variable suit, **dans une classe**, une **gaussienne**
$\mathcal{N}(\mu_{cj}, \sigma_{cj}^2)$. Mais une variable reelle peut etre **asymetrique** ou **bimodale** :
une seule cloche gaussienne la represente alors mal. L'**estimation par noyau** (*kernel density estimation*,
KDE) remplace cette hypothese parametrique par une estimation **non parametrique** de la densite :
$$\hat f_h(x) \;=\; \frac{1}{n\,h}\sum_{i=1}^{n} K\!\left(\frac{x - x_i}{h}\right),$$
ou $K$ est un **noyau** (fonction positive, symetrique, d'integrale 1) et $h > 0$ la **fenetre** (*bandwidth*).
On depose une petite bosse $K$ sur chaque observation, puis on les somme.

### A quoi ca ressemble ?

Noyaux courants (vus en cours) :
$$K_{\text{gaussien}}(t) = \tfrac{1}{\sqrt{2\pi}} e^{-t^2/2}, \qquad
  K_{\text{Epanechnikov}}(t) = \tfrac{3}{4}(1 - t^2)\,\mathbb{1}_{|t|\le 1}.$$
Le role de la **fenetre $h$** est celui du $k$ des plus proches voisins : $h$ **petit** ⇒ on lisse peu
(variance forte) ; $h$ **grand** ⇒ on lisse beaucoup (biais fort). On choisit $h$ par **validation croisee**.

### Le lien avec les hyperparametres du Naive Bayes en R

C'est precisement ce que reglent les hyperparametres de la methode `nb` de `caret` :
- **`usekernel`** : utiliser une densite par **noyau** (KDE) au lieu de la gaussienne ;
- **`adjust`** : un multiplicateur de la **fenetre $h$** (plus ou moins de lissage) ;
- **`fL`** : la **correction de Laplace** (pour les variables discretes, evite les probabilites nulles).

### Quel interet en pratique ?

Quand la vraie densite conditionnelle est **loin d'une gaussienne**, la KDE l'estime bien mieux ⇒ un Naive Bayes
« a noyau » est **plus fidele** et souvent **plus predictif**. Le prix a payer : un hyperparametre de plus ($h$)
a regler. L'illustration ci-dessous compare, pour une variable de vin, l'ajustement **gaussien** et l'estimation
**par noyau**, puis montre l'effet de la fenetre $h$.


In [14]:
# Exemple concret : densite de 'alcohol' chez les bons vins, gaussienne vs noyau
vals <- X$alcohol[y == "bon"]
g <- seq(min(vals), max(vals), length.out = 200)
dens_gauss <- dnorm(g, mean(vals), sd(vals))
kde <- density(vals, from = min(vals), to = max(vals), n = 200)

p1 <- plot_ly() %>%
  add_trace(x = vals, type = "histogram", histnorm = "probability density",
            nbinsx = 30, name = "histogramme",
            marker = list(color = MUTED), opacity = 0.5) %>%
  add_trace(x = g, y = dens_gauss, type = "scatter", mode = "lines",
            name = "hypothese gaussienne", line = list(color = pick_color(1), width = 3)) %>%
  add_trace(x = kde$x, y = kde$y, type = "scatter", mode = "lines",
            name = "noyau (KDE)", line = list(color = pick_color(0), width = 3)) %>%
  layout(xaxis = list(title = "alcohol"), yaxis = list(title = "densite"))

# Effet de la fenetre h (hyperparametre 'adjust')
mults <- c(0.3, 1, 2.5); labs <- c("h petit", "h defaut", "h grand")
p2 <- plot_ly()
for (i in seq_along(mults)) {
  d <- density(vals, adjust = mults[i], from = min(vals), to = max(vals), n = 200)
  p2 <- add_trace(p2, x = d$x, y = d$y, type = "scatter", mode = "lines",
                  name = labs[i], line = list(color = pick_color(i - 1), width = 2))
}
p2 <- layout(p2, xaxis = list(title = "alcohol"), yaxis = list(title = "densite"))

subplot(p1, p2, nrows = 1, titleX = TRUE, margin = 0.07) %>%
  layout(title = "Estimation par noyau de la densite de 'alcohol' (bons vins)") %>%
  apply_dark_theme(height = 420)

HTML widgets cannot be represented in plain text (need html)

---

## Annexe G — Les hyperparametres de chaque modele : role et autres reglages possibles

Un **hyperparametre** n'est pas appris sur les donnees : on le **fixe avant** l'entrainement (ici par validation
croisee). Voici, modele par modele, **a quoi correspond** chaque hyperparametre et **quels autres** on pourrait
regler en R.

### Naive Bayes (`caret`, methode `nb` -> `klaR::NaiveBayes`)
- **`usekernel`** *(regle ici)* : si `TRUE`, la densite de chaque variable est estimee par **noyau** (KDE) au
  lieu d'une gaussienne (cf Annexe F) ; si `FALSE`, on garde la gaussienne.
- **`adjust`** *(regle ici)* : multiplicateur de la **fenetre** $h$ du noyau (plus grand = plus lisse).
- **`fL`** *(regle ici)* : **correction de Laplace** (evite les probabilites nulles, surtout pour des variables
  discretes).
- *Autre* : on peut fixer les probabilites a priori des classes.

### LDA / QDA (`MASS::lda` / `MASS::qda`)
- **Pas d'hyperparametre a proprement parler** : tout est fixe par les **hypotheses** (gaussiennes, covariances
  egales pour LDA, libres pour QDA).
- *Autres reglages disponibles* : **`prior`** (probabilites a priori des classes) et **`method`** = mode
  d'estimation des parametres : `"moment"` (defaut), `"mle"`, `"mve"` (estimation **robuste**) ou `"t"` (loi de
  Student, robuste aux valeurs extremes). On peut aussi **regulariser** entre LDA et QDA via `klaR::rda`.

### k plus proches voisins (`caret`, methode `knn`)
- **`k`** *(regle ici)* : nombre de voisins. Petit `k` ⇒ sur-apprentissage ; grand `k` ⇒ sous-apprentissage.
- *Autres* : pour **ponderer** les voisins par la distance ou changer de **metrique**, on utilise la methode
  `kknn` (paquet `kknn`), qui expose `kmax`, `distance` (Minkowski) et `kernel` (ponderation).
- Le k-NN raisonnant par distances, on le fait **toujours** preceder d'une **standardisation**
  (`preProcess = c("center", "scale")`).

> **A retenir.** LDA, QDA et Naive Bayes gaussien ont **peu** d'hyperparametres, car l'essentiel de leur
> comportement vient de leurs **hypotheses de modele**. Le k-NN, non parametrique, se regle surtout via `k`.


---

## Annexe H — Pourquoi optimiser l'**AUC** plutot que l'**accuracy** ?

On utilise l'**AUC** (aire sous la courbe ROC) — et non l'accuracy — aussi bien pour **regler les
hyperparametres** (validation croisee) que pour **comparer** les modeles sur le test. Trois raisons.

**1. Les classes sont desequilibrees.** Seuls ~14 % des vins sont « bons ». Un classifieur trivial qui predirait
**toujours « mauvais »** obtiendrait deja ~86 % d'accuracy **sans rien apprendre**. L'accuracy recompense donc la
classe majoritaire et **masque** l'incapacite a reperer les bons vins. L'AUC, elle, est **insensible** a la
proportion des classes.

**2. L'accuracy depend d'un seuil arbitraire.** Elle fige la decision au **seuil $1/2$**. Or un bon modele peut
tres bien **ordonner** les vins (scores eleves pour les bons) tout en ayant un seuil optimal different de $1/2$.
L'AUC evalue la qualite de cet **ordonnancement** pour **tous les seuils** a la fois.

**3. Interpretation probabiliste claire.** L'AUC est **la probabilite qu'un bon vin tire au hasard recoive un
score plus eleve qu'un mauvais vin tire au hasard** :
$$\text{AUC} = \mathbb{P}\big(\hat{s}(X^{+}) > \hat{s}(X^{-})\big), \qquad X^{+}\ \text{bon},\ X^{-}\ \text{mauvais}.$$
Une AUC de $0.5$ correspond au hasard, $1.0$ a un classement parfait.

**Pour le reglage des hyperparametres**, optimiser l'AUC evite de choisir des reglages qui se contentent de
predire la classe majoritaire. **Pour la comparaison finale**, elle offre une mesure **equitable** et
**independante du seuil**. On conserve tout de meme l'**accuracy** comme metrique **complementaire** et
facilement interpretable.
